In [1]:
# ==============================================================================
# === 0 INSTALLATION CORRECTIONS
# ==============================================================================
!pip uninstall -y protobuf
!pip install protobuf==3.20.3

Found existing installation: protobuf 5.29.5
Uninstalling protobuf-5.29.5:
  Successfully uninstalled protobuf-5.29.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
onnx 1.20.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
ray 2.52.1 requires click!=8.3.*,>=7.0, but you have click 8.3.1 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
ydf 0.13.0 requires 

In [2]:
# ==============================================================================
# === 1.1 INSTALLATIONS
# ==============================================================================
print("Installing required packages...")
!pip install sentence-transformers -q
!pip install transformers accelerate -q
!pip install scikit-learn seaborn -q
print("Installations complete.")

Installing required packages...
Installations complete.


In [3]:
# ==============================================================================
# === 1.2 IMPORTS
# ==============================================================================
import re
import os
import logging
import pandas as pd
import numpy as np
import torch
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# For Semantic Textual Similarity (STS)
# from sentence_transformers import SentenceTransformer, util

# For Natural Language Inference (NLI)
from transformers import pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

2025-12-21 07:11:22.264499: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766301082.462224      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766301082.525142      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766301083.075741      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766301083.075782      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766301083.075785      24 computation_placer.cc:177] computation placer alr

In [4]:
# ==============================================================================
# === 2. CONFIGURATION & DATA LOADING
# ==============================================================================

class AnalysisConfig:
    POS_DATA_FILE = "/kaggle/input/lrec-tcs-attack-support/sentencePair.txt"
    NEG_DATA_FILE = "/kaggle/input/lrec-tcs-attack-support/sentencePair_neg.txt"
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    NLI_SAMPLE_SIZE = 2000 # NLI is slow, so we sample.
    NLI_MODEL = 'facebook/bart-large-mnli' # Good zero-shot NLI model
    OUTFILE1 = "/kaggle/working/bart_large_mnli_zeroshot.csv"
    OUTFILE2 = "/kaggle/working/bart_large_mnli_explicit_nli.csv"

# Setup basic logging
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
if not logger.handlers:
    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)

def parse_lrec_line(line: str):
    """Parses a single line from the LREC dataset files."""
    parts = line.strip().split("\t")
    fname_indices = [i for i, p in enumerate(parts) if p.endswith(".txt")]
    if len(fname_indices) != 2: return None
    try:
        sentpair_id = int(parts[0])
        sent1 = " ".join(parts[fname_indices[0] + 2 : fname_indices[1]]).strip('"')
        sent2 = " ".join(parts[fname_indices[1] + 2 : len(parts) - 2]).strip('"')
        label = parts[-1]
    except (ValueError, IndexError): return None
    return {"id": sentpair_id, "sent1": sent1, "sent2": sent2, "label": label}

def load_full_data(config: AnalysisConfig):
    """Loads and parses the full dataset from both positive and negative files."""
    logging.info("Loading full dataset for analysis...")
    rows = []
    for filepath in [config.POS_DATA_FILE, config.NEG_DATA_FILE]:
        if not os.path.exists(filepath):
            logging.error(f"Data file not found at '{filepath}'.")
            return None
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                if parsed := parse_lrec_line(line): rows.append(parsed)
    df = pd.DataFrame(rows)
    label_map = {"SUPPORT": 0, "ATTACK": 1, "NO_REL": 2}
    df["label_id"] = df["label"].map(label_map)
    df = df.dropna(subset=['id', 'sent1', 'sent2', 'label_id', 'label'])
    logging.info(f"Full dataset loaded. Total samples: {len(df)}")
    return df.reset_index(drop=True)

In [5]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1

zeroshot_clf = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [6]:
def bart_zeroshot_classify(sent1, sent2):
    premise = f"""
Sentence 1: {sent1}
Sentence 2: {sent2}
"""

    labels = ["SUPPORT", "ATTACK", "NO_REL"]

    result = zeroshot_clf(
        premise,
        candidate_labels=labels,
        hypothesis_template="The relationship between the two sentences is {}."
    )

    return result["labels"][0], result["scores"][0]


In [7]:
config = AnalysisConfig()
full_df = load_full_data(config)
df_sample = full_df.sample(n=config.NLI_SAMPLE_SIZE, random_state=42)

In [8]:
from sklearn.metrics import classification_report

df_sample["true_label_norm"] = (
    df_sample["label"]
    .str.upper()
    .str.replace(" ", "_")
    .replace({"NO_RELATION": "NO_REL"})
)

In [9]:
preds = []
scores = []

for _, row in df_sample.iterrows():
    label, score = bart_zeroshot_classify(row.sent1, row.sent2)
    preds.append(label)
    scores.append(score)

df_sample["bart_zs_label"] = preds
df_sample["bart_zs_score"] = scores

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [10]:
print("\n=== BART MNLI Zero-Shot: Crosstab ===")
crosstab_zs = pd.crosstab(
    df_sample["true_label_norm"],
    df_sample["bart_zs_label"],
    rownames=["True Label"],
    colnames=["Predicted Label"]
)
print(crosstab_zs.to_markdown())


=== BART MNLI Zero-Shot: Crosstab ===
| True Label   |   ATTACK |   NO_REL |   SUPPORT |
|:-------------|---------:|---------:|----------:|
| ATTACK       |      224 |       66 |       200 |
| NO_REL       |      360 |      122 |       483 |
| SUPPORT      |      185 |       66 |       294 |


In [11]:
print("\n=== BART MNLI Zero-Shot: Classification Report ===")
print(
    classification_report(
        df_sample["true_label_norm"],
        df_sample["bart_zs_label"],
        labels=["SUPPORT", "ATTACK", "NO_REL"],
        zero_division=0
    )
)


=== BART MNLI Zero-Shot: Classification Report ===
              precision    recall  f1-score   support

     SUPPORT       0.30      0.54      0.39       545
      ATTACK       0.29      0.46      0.36       490
      NO_REL       0.48      0.13      0.20       965

    accuracy                           0.32      2000
   macro avg       0.36      0.37      0.31      2000
weighted avg       0.39      0.32      0.29      2000



In [12]:
bart_zs_out = df_sample[[
    "id",
    "sent1",
    "sent2",
    "label",
    "bart_zs_label",
    "bart_zs_score"
]].rename(columns={
    "label": "true_label"
})

bart_zs_out.to_csv(
    "/kaggle/working/bart_large_mnli_zeroshot.csv",
    index=False
)

print("Saved BART zero-shot results to bart_large_mnli_zeroshot.csv")

Saved BART zero-shot results to bart_large_mnli_zeroshot.csv


In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-mnli")
model = AutoModelForSequenceClassification.from_pretrained(
    "facebook/bart-large-mnli"
).to(device)

model.eval()


BartForSequenceClassification(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [14]:
def bart_explicit_nli(sent1, sent2):
    inputs = tokenizer(
        sent1,
        sent2,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.softmax(logits, dim=1)[0]

    # MNLI label order for BART:
    # 0 = contradiction, 1 = neutral, 2 = entailment
    label_id = torch.argmax(probs).item()

    id2label = {0: "contradiction", 1: "neutral", 2: "entailment"}
    return id2label[label_id], probs[label_id].item(), probs.cpu().tolist()

In [15]:
def map_nli_to_argument(nli_label):
    if nli_label == "entailment":
        return "SUPPORT"
    elif nli_label == "contradiction":
        return "ATTACK"
    else:
        return "NO_REL"

In [16]:
nli_labels = []
argument_labels = []
nli_scores = []

for _, row in df_sample.iterrows():
    nli_label, score, _ = bart_explicit_nli(row.sent1, row.sent2)
    arg_label = map_nli_to_argument(nli_label)

    nli_labels.append(nli_label)
    argument_labels.append(arg_label)
    nli_scores.append(score)

df_sample["bart_nli_label"] = nli_labels
df_sample["bart_argument_label"] = argument_labels
df_sample["bart_nli_score"] = nli_scores

In [17]:
print("\n=== BART MNLI Explicit NLI: Crosstab ===")
crosstab_nli = pd.crosstab(
    df_sample["true_label_norm"],
    df_sample["bart_argument_label"],
    rownames=["True Label"],
    colnames=["Predicted Label"]
)
print(crosstab_nli.to_markdown())


=== BART MNLI Explicit NLI: Crosstab ===
| True Label   |   ATTACK |   NO_REL |   SUPPORT |
|:-------------|---------:|---------:|----------:|
| ATTACK       |      192 |      272 |        26 |
| NO_REL       |      241 |      706 |        18 |
| SUPPORT      |      113 |      385 |        47 |


In [18]:
print("\n=== BART MNLI Explicit NLI: Classification Report ===")
print(
    classification_report(
        df_sample["true_label_norm"],
        df_sample["bart_argument_label"],
        labels=["SUPPORT", "ATTACK", "NO_REL"],
        zero_division=0
    )
)


=== BART MNLI Explicit NLI: Classification Report ===
              precision    recall  f1-score   support

     SUPPORT       0.52      0.09      0.15       545
      ATTACK       0.35      0.39      0.37       490
      NO_REL       0.52      0.73      0.61       965

    accuracy                           0.47      2000
   macro avg       0.46      0.40      0.37      2000
weighted avg       0.48      0.47      0.42      2000



In [19]:
bart_nli_out = df_sample[[
    "id",
    "sent1",
    "sent2",
    "label",
    "bart_nli_label",
    "bart_argument_label",
    "bart_nli_score"
]].rename(columns={
    "label": "true_label"
})

bart_nli_out.to_csv(
    "/kaggle/working/bart_large_mnli_explicit_nli.csv",
    index=False
)

print("Saved BART explicit NLI results to bart_large_mnli_explicit_nli.csv")


Saved BART explicit NLI results to bart_large_mnli_explicit_nli.csv
